In [13]:
import numpy as np
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional

In [14]:
"""
SBTI - Hybrid Calculation Method 
1. Không gian Vector 15 chiều
2. Reverse Scoring + Weighted Accumulation
3. Mahalanobis Distance (statistically superior)
4. Pattern Detection + Dimension Analysis
5. 27 Archetypes
"""

'\nSBTI - Hybrid Calculation Method \n1. Không gian Vector 15 chiều\n2. Reverse Scoring + Weighted Accumulation\n3. Mahalanobis Distance (statistically superior)\n4. Pattern Detection + Dimension Analysis\n5. 27 Archetypes\n'

In [15]:
# ====================== DIMENSIONS ======================
DIMENSION_NAMES = [
    "S1_SelfEsteem", "S2_SelfClarity", "S3_Purpose",
    "E1_Attachment", "E2_EmotionalDepth", "E3_Independence",
    "A1_Worldview", "A2_RulesFlex", "A3_Meaning",
    "Ac1_Motivation", "Ac2_Decision", "Ac3_Execution",
    "So1_SocialProactivity", "So2_Boundaries", "So3_Authenticity"
]

DIMENSION_GROUPS = {
    "Bản Thân": [0, 1, 2],
    "Cảm Xúc": [3, 4, 5],
    "Thái Độ": [6, 7, 8],
    "Hành Động": [9, 10, 11],
    "Xã Hội": [12, 13, 14]
}

# ====================== DIMENSION WEIGHTS ======================
DIMENSION_WEIGHTS = np.array([
    1.25, 1.30, 1.35, 1.20, 1.40, 1.15, 1.10, 1.05, 
    1.20, 1.50, 1.55, 1.60, 1.10, 1.25, 1.15
])
DIMENSION_WEIGHTS = DIMENSION_WEIGHTS / np.sum(DIMENSION_WEIGHTS)

# ====================== 27 ARCHETYPES ======================
ARCHETYPES = {
    "CTRL":  {"vector": [2,2,2,1,1,1,2,2,1,2,2,2,2,1,1], "desc": "The Controller - Master of everything", "code": "CTRL"},
    "BOSS":  {"vector": [2,1,2,1,2,1,2,1,2,2,2,2,2,1,2], "desc": "The Boss - Natural leader", "code": "BOSS"},
    "MALO":  {"vector": [0,0,0,0,0,2,0,0,0,0,0,0,1,2,0], "desc": "MALO - Professional Slacker", "code": "MALO"},
    "DEAD":  {"vector": [0,1,0,0,0,2,1,1,0,0,0,0,0,2,0], "desc": "DEAD - Emotionally Dead", "code": "DEAD"},
    "ATMR":  {"vector": [1,1,1,2,2,0,1,1,1,1,1,1,2,0,1], "desc": "ATMR - Walking Wallet", "code": "ATMR"},
    "IMSB":  {"vector": [0,2,1,1,2,1,1,1,2,0,1,0,1,1,2], "desc": "IMSB - Self-roasting Champion", "code": "IMSB"},
    "SHIT":  {"vector": [1,1,0,0,1,2,0,0,0,1,0,1,0,2,1], "desc": "SHIT - Professional Hater", "code": "SHIT"},
    "SOLO":  {"vector": [1,2,1,0,0,2,1,2,1,1,1,1,0,2,2], "desc": "SOLO - Lone Wolf", "code": "SOLO"},
    "DRNK":  {"vector": [1,1,1,2,2,0,0,0,2,0,0,0,2,0,2], "desc": "DRNK - Chaos Gremlin", "code": "DRNK"},
    "OHNO":  {"vector": [1,1,1,2,2,1,2,0,1,1,0,1,1,1,1], "desc": "OHNO - Professional Worrier", "code": "OHNO"},
    "GOGO":  {"vector": [2,1,2,1,1,1,1,1,1,2,2,2,2,1,1], "desc": "GOGO - Hyperactive Doer", "code": "GOGO"},
    "FAKE":  {"vector": [1,1,1,1,1,1,1,2,1,1,1,1,2,1,0], "desc": "FAKE - Social Chameleon", "code": "FAKE"},
    "LOVR":  {"vector": [1,2,2,2,2,0,1,1,2,1,1,1,2,0,2], "desc": "LOVR - Romantic Maximalist", "code": "LOVR"},
    "MUMM":  {"vector": [2,1,2,2,2,1,2,2,1,1,1,1,1,1,1], "desc": "MUMM - Emotional Support Human", "code": "MUMM"},
    "HHHH":  {"vector": [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1], "desc": "HHHH - Ultimate Chaos Entity", "code": "HHHH"},
    "NICE":  {"vector": [1,1,2,2,1,1,2,1,2,1,1,1,2,1,1], "desc": "NICE - Too Nice For This World", "code": "NICE"},
    "TOXC":  {"vector": [1,0,1,0,2,2,0,0,1,1,0,2,1,2,0], "desc": "TOXC - Walking Red Flag", "code": "TOXC"},
    "CHAD":  {"vector": [2,2,2,1,1,2,2,2,2,2,2,2,2,2,2], "desc": "CHAD - Sigma Male Grinder", "code": "CHAD"},
    "DOOM":  {"vector": [0,0,0,1,0,0,0,1,0,0,0,0,0,1,0], "desc": "DOOM - Professional Doomer", "code": "DOOM"},
    "MEME":  {"vector": [1,2,1,1,1,1,1,0,2,1,1,0,2,1,2], "desc": "MEME - Internet Gremlin", "code": "MEME"},
    "SIMPP":{"vector": [1,1,1,2,2,0,1,1,1,0,0,0,2,0,1], "desc": "SIMPP - Professional Simp", "code": "SIMPP"},
    "GIGI":  {"vector": [2,1,2,1,1,2,2,1,1,2,2,1,1,2,1], "desc": "GIGI - Girlboss / Guyboss", "code": "GIGI"},
    "ZEN":   {"vector": [2,2,2,0,1,2,2,2,2,1,1,2,1,2,2], "desc": "ZEN - Enlightened Buddha", "code": "ZEN"},
    "CLWN":  {"vector": [1,1,1,2,2,1,1,0,1,0,1,0,2,1,1], "desc": "CLWN - Professional Clown", "code": "CLWN"},
    "SAD":   {"vector": [0,2,0,2,2,0,1,1,0,0,0,0,0,1,2], "desc": "SAD - Sad Boi Hours", "code": "SAD"},
    "HUST":  {"vector": [2,1,2,1,1,1,2,2,2,2,2,2,1,1,1], "desc": "HUST - 24/7 Hustler", "code": "HUST"},
    "VOID":  {"vector": [0,0,1,0,0,2,1,1,1,0,0,0,0,2,0], "desc": "VOID - Existential Crisis", "code": "VOID"},
}

WEIGHTS = DIMENSION_WEIGHTS  # alias for compatibility

In [ ]:
# ====================== MATCHING SETUP ======================
# Logic:
# 1. User answers 30 questions (2 per dimension)
# 2. Average scores per dimension (0-2 range)
# 3. Calculate distance to each archetype
# 4. Convert distance to similarity %
# 5. Rank archetypes by similarity
# 6. Calculate confidence based on score gap

✓ Simplified matching engine initialized (Euclidean distance)


In [ ]:
class SBTI_Hybrid:
    def __init__(self):
        self.names = list(ARCHETYPES.keys())
        self.vectors = np.array([ARCHETYPES[n]["vector"] for n in self.names])

    def get_questions(self):
        return [
            {"dim":0, "q":"Trong chuyện tình cảm, tôi thường cảm thấy mình không đủ tốt so với những người yêu cũ của người ấy.", "opts":["Đúng vậy, tôi hay bị ám ảnh bởi điều đó.", "Thỉnh thoảng tôi mới nghĩ vậy khi mọi thứ không suôn sẻ.", "Không, tôi tin vào giá trị của mình ở hiện tại."], "reverse": False},
            {"dim":0, "q":"Người yêu bạn rủ bạn về ra mắt gia đình, nhưng bạn biết gia đình họ có điều kiện và rất khó tính.", "opts":["Sợ mình không đủ tốt, kiểu gì cũng bị chê.", "Tôi lo một chút, chuẩn bị trước vài thứ để tự tin hơn.", "Tôi tự tin mình sẽ ghi điểm."], "reverse": True},
            {"dim":1, "q":"Tôi là ai? Tôi thích gì? Tôi giỏi gì? Tôi muốn gì? Đừng hỏi tôi...", "opts":["Ủa, sao giống mình thế...", "Mình đang đọc cái * thế này ... ?", "Tôi có như này đâu !?"], "reverse": False},
            {"dim":1, "q":"Bạn và người yêu đang cãi nhau. Giữa lúc căng thẳng, bạn có biết chính xác mình đang giận vì điều gì không?", "opts":["Chỉ thấy khó chịu và muốn người kia im mồm luôn", "Tôi cần vài phút im lặng để tự hỏi", "Biết vì sao mình giận và nói rõ"], "reverse": True},
            {"dim":2, "q":"Nếu ai cũng biết 'kệ' nhau để sống thoải mái, mục tiêu lớn nhất của bạn là gì?", "opts":["Có lẽ cứ sống theo cách người khác mong đợi.", "Chưa dám khẳng định điều gì là quan trọng nhất với mình.", "Tôi biết rõ mình coi trọng điều gì, và tôi sẽ sống hết mình vì điều đó."], "reverse": False},
            {"dim":2, "q":"Tôi thà bị ghét vì là chính mình, còn hơn được yêu quý vì là bản sao của ai khác.", "opts":["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": True},
            {"dim":3, "q":"Tôi thường cảm thấy bất an và nghi ngờ khi ai đó đối xử tốt với mình một cách bất ngờ.", "opts":["Đồng ý", "Trung lập", "Không đồng ý"], "reverse": False},
            {"dim":3, "q":"Trong các lần làm bài tập nhóm, khi gặp bất đồng quan điểm thì bạn sẽ làm gì?", "opts":["Thôi im lặng cho yên, sợ cãi nhau xong nghỉ chơi luôn thì sao?!", "Bảo vệ luận điểm của mình và không làm quá mọi chuyện.", "Cãi thắng bằng được!"], "reverse": True},
            {"dim":4, "q":"Bạn mới quen một người. Họ tâm sự với bạn về một tổn thương sâu sắc trong quá khứ.", "opts":["Gì vậy ta, mới quen á má!", "Bạn lắng nghe chăm chú, an ủi họ nhưng vẫn giữ khoảng cách.", "Đồng cảm và chia sẻ ngay những trải nghiệm tương tự của bản thân."], "reverse": False},
            {"dim":4, "q":"Tôi thấy khó mở lòng và thường mất nhiều thời gian mới thực sự tin tưởng ai đó.", "opts":["Không đồng ý.", "Trung lập.", "Đồng ý."], "reverse": True},
            {"dim":5, "q":"Người yêu bạn hào hứng muốn kể cho bạn nghe về một ngày dài của họ. Bạn sẽ?", "opts":["Dừng việc đang làm, tập trung lắng nghe.", "Vừa nghe vừa làm việc của mình, thỉnh thoảng gật gù.", "Từ chối khéo vì đang bận việc riêng."], "reverse": False},
            {"dim":5, "q":"Khi mới chuyển lớp, đã có một người bạn ra bắt chuyện với tôi một cách rất thân thiện.", "opts":["Nói chuyện lại như người nhà.", "Chào hỏi xã giao.", "Im lặng nào, thầy đang mewing !!"], "reverse": False},
            {"dim":6, "q":"Bạn thấy một người cùng tuổi bạn có kỹ năng tốt hơn, đang kiếm tiền / làm việc tốt hơn bạn:", "opts":["Bạn nghĩ họ có nền tảng hoặc xuất phát điểm tốt hơn nên mình có cố cũng khó theo kịp", "Bạn nghĩ mỗi người có hoàn cảnh và hướng đi khác nhau", "Tìm hiểu xem họ có bí quyết hay cách làm gì khác mình"], "reverse": False},
            {"dim":6, "q":"Bạn đăng một thứ bạn làm (vẽ tranh, blog, video…), và có người vào góp ý khá thẳng:", "opts":["Thấy khó chịu và muốn gỡ bài xuống.", "Đọc xong để đó, không suy nghĩ nhiều.", "Tiếp thu ý kiến và cân nhắc chỉnh sửa"], "reverse": True},
            {"dim":7, "q":"Cách bạn sắp xếp không gian sống hoặc phòng riêng của mình:", "opts":["Sống trong sự hỗn độn có chủ đích", "Tổng quan gọn gàng, nhưng đôi khi có những góc bừa bộn riêng", "Mọi thứ phải nằm đúng vị trí của nó"], "reverse": False},
            {"dim":7, "q":"Sắp tới bạn có một chuyến du lịch nhưng bạn chưa có lịch trình cụ thể.", "opts":["Đi rồi tính, tới đâu hay tới đó mới vui.", "Lên vài điểm chính muốn đến, còn lại tùy hứng", "Đã lên chi tiết từng khung giờ"], "reverse": True},
            {"dim":8, "q":"Nếu có một nút bấm cho phép bạn xem trước tương lai nhưng không thể thay đổi, bạn sẽ làm gì?", "opts":["Bấm luôn, xem khi nào trúng số", "Suy nghĩ một hồi...", "Không bấm, vì tương lai là của mình là tự viết"], "reverse": False},
            {"dim":8, "q":"Bạn bất ngờ nhận được một khoản tiền từ trên trời rơi xuống. Phản ứng đầu tiên của bạn là gì?", "opts":["Chốt ngay cho bản thân một món đồ mình đã tia lâu nay", "Cất đi trước rồi tính sau", "Tận dụng làm đòn bẩy. Đầu tư ngay vào một khóa học"], "reverse": True},
            {"dim":9, "q":"Bạn đang đi trên phố thì thấy một chiếc xe hơi đang lao xuống dốc, phía dưới là một em bé đang chơi.", "opts":["Hét lên, hy vọng có người khác đến cứu.", "Vừa chạy vừa tính toán cách đẩy em bé ra", "Lao ngay vào, chấp nhận rủi ro"], "reverse": False},
            {"dim":9, "q":"Một con quỷ xuất hiện trước mặt bạn và thì thầm: 'Cho tao một phần cơ thể của mày...'", "opts":["Điên à? Mất một phần cơ thể thì còn gì là mình nữa.", "Từ từ, để coi nó lấy phần nào đã.", "Lấy đi! Chỉ cần tao mạnh hơn tất cả!"], "reverse": False},
            {"dim":10,"q":"Bạn bỗng nhận ra bài test này đang đo lường chính sự do dự của bạn. Bạn càng do dự, điểm càng thấp. Bạn sẽ:", "opts":["Chết rồi. Giờ biết chọn sao đây?!", "Hay thật đấy. Để tao cân nhắc thêm một chút...", "Biết rồi, chọn luôn!"], "reverse": True},
            {"dim":10,"q":"Bạn và nhóm bạn bị kẹt trong một tòa nhà đầy zombie...", "opts":["Đứng im, chờ xem ai trong nhóm quyết định trước", "Đánh giá nhanh tình hình", "Hô to 'Lên mái nhà!' và lao đi trước."], "reverse": False},
            {"dim":11,"q":"23h đêm, tự dưng đứa bạn nhắc mai là hạn deadline quan trọng. Bạn làm gì?", "opts":["Hoảng loạn, tìm cách gia hạn deadline.", "Cà phê, bật nhạc lofi, ngồi vào bàn và làm tới sáng", "Bình tĩnh, vì đã làm xong từ tuần trước"], "reverse": False},
            {"dim":11,"q":"Tôi thường trì hoãn những việc quan trọng cho đến khi không thể trì hoãn được nữa.", "opts":["Đồng ý", "Trung lập", "Không đồng ý"], "reverse": True},
            {"dim":12,"q":"Khi tham gia 1 nhóm mới hoặc gặp người lạ, bạn sẽ:", "opts":["E ngại, thường đứng ngoài quan sát.", "Tùy người. Hợp thì chơi.", "Bạn của bạn cũng là bạn tôi!"], "reverse": False},
            {"dim":12,"q":"Trong một nhóm chat đông người đang tranh luận sôi nổi, vai trò của bạn là gì?", "opts":["Để chế độ im lặng", "Thỉnh thoảng vào thả icon", "Chiến thần spam tin nhắn"], "reverse": False},
            {"dim":13,"q":"Một người bạn tự ý lấy đồ của bạn mà không xin phép, bạn sẽ:", "opts":["Thấy khó chịu trong lòng, nhưng vẫn im lặng", "Nhắc nhở họ khéo và xin lại đồ.", "Tỏ thái độ gay gắt, giật lại đồ"], "reverse": False},
            {"dim":13,"q":"Người yêu bạn yêu cầu bạn hạn chế chơi với bạn thân khác giới. Bạn sẽ:", "opts":["Đồng ý ngay vì không muốn người yêu buồn lòng.", "Cố gắng giải thích...", "Khẳng định rằng bạn bè và tình yêu là hai phạm trù riêng"], "reverse": True},
            {"dim":14,"q":"Khi ở cạnh người khác, bạn thể hiện bản thân thế nào?", "opts":["Khéo léo thay đổi cách nói chuyện theo từng hoàn cảnh", "Có lúc thẳng thắn, có lúc tiết chế", "Thể hiện đúng con người thật"], "reverse": False},
            {"dim":14,"q":"Tôi có thể dễ dàng thay đổi giọng điệu, cách nói và thậm chí cả vốn từ tùy theo người tôi đang trò chuyện.", "opts":["Đồng ý", "Trung lập", "Không đồng ý"], "reverse": True},
            {"dim":-1, "q":"Câu hỏi bonus: Thói quen uống rượu / say xỉn của bạn?", "opts":["0 → Thánh Say - Uống vào là thăng hoa cực độ", "1 → Xã giao - Uống có chừng mực", "2 → Không uống hoặc rất ít"], "reverse": False}
        ]

    def take_test(self):
        print("=== SBTI - Hybrid Method v2.4 (Improved Scoring) ===\n")
        dim_answers = [[] for _ in range(15)]
        bonus_drink = 1
        questions = self.get_questions()
        
        for i, q in enumerate(questions):
            print("\n" + "="*90)
            print(f"QUESTION {i+1:2d} / {len(questions)}")
            print("="*90)
            print(q['q'] + "\n")
            for j, opt in enumerate(q['opts']):
                print(f"   {j} - {opt}")
            print("-"*90)
            
            while True:
                try:
                    ans = int(input("Enter choice (0/1/2): ").strip())
                    if 0 <= ans <= 2:
                        if q['dim'] >= 0:
                            # Convert answer to score (0-2 range)
                            score = (2 - ans) if q.get('reverse', False) else ans
                            dim_answers[q['dim']].append(score)
                            print(f"   OK - Saved for {DIMENSION_NAMES[q['dim']]}")
                        else:
                            bonus_drink = ans
                        break
                    else:
                        print("   Enter 0, 1, or 2 only.")
                except:
                    print("   Please enter a valid number.")
        
        # Average scores per dimension (results in 0.0-2.0 range)
        user_vector = []
        for d in range(15):
            if dim_answers[d]:
                user_vector.append(round(np.mean(dim_answers[d]), 3))
            else:
                user_vector.append(1.0)
        return np.array(user_vector), bonus_drink

    def distance_match(self, user_vector: np.ndarray) -> tuple:
        """
        Simple Euclidean distance (honest scoring)
        
        Logic:
        1. Calculate euclidean distance between user and each archetype
        2. Convert distance to similarity % using exponential decay
        3. Calculate confidence based on gap between top 2 scores
        """
        results = []
        user = np.array(user_vector)
        
        for name in self.names:
            arch = np.array(ARCHETYPES[name]["vector"], dtype=float)
            
            # Euclidean distance: sqrt(sum of squared differences)
            diff = user - arch
            dist = np.sqrt(np.sum(diff ** 2))
            
            # Convert to similarity % (exponential decay)
            # Using decay factor 4.5 so ranges are reasonable (0-100%)
            similarity = max(0, round(100 * np.exp(-dist / 4.5), 1))
            
            results.append({
                "type": name,
                "code": ARCHETYPES[name]["code"],
                "score": similarity,
                "dist": round(dist, 3),
                "desc": ARCHETYPES[name]["desc"]
            })
        
        results = sorted(results, key=lambda x: x["score"], reverse=True)
        
        # Calculate confidence based on score gap
        top_score = results[0]["score"]
        second_score = results[1]["score"] if len(results) > 1 else 0
        gap = top_score - second_score
        
        if gap > 15:
            confidence = "HIGH"
        elif gap > 5:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"
        
        return results, confidence

    def analyze_dimensions(self, scores: np.ndarray) -> Dict:
        analysis = {}
        for group_name, indices in DIMENSION_GROUPS.items():
            group_scores = scores[list(indices)]
            avg = np.mean(group_scores)
            
            # Level classification
            if avg >= 1.33:
                level = "HIGH"
            elif avg >= 0.67:
                level = "MEDIUM"
            else:
                level = "LOW"
            
            analysis[group_name] = {
                "level": level,
                "score": round(avg * 50, 1),  # Scale to 0-100
                "raw": [round(x, 2) for x in group_scores]
            }
        return analysis

In [19]:
# ==================== MAIN EXECUTION ====================
sbti = SBTI_Hybrid()
user_vector, bonus_drink = sbti.take_test()

print("\n" + "="*90)
print("ANALYZING YOUR RESULTS (Improved Euclidean Matching)...")
print("="*90)

results, confidence = sbti.distance_match(user_vector)

# Get final result
final_type = results[0]["type"]
final_score = results[0]["score"]

# ==================== DISPLAY RESULTS ====================
print("\n" + "="*90)
print("YOUR MAIN TYPE")
print("="*90)
print(f"\n{final_type} ({final_score:.1f}%)")
print(ARCHETYPES[final_type]["desc"])
print(f"Confidence: {confidence}")
print(f"Method: Simple Euclidean Distance\n")

print("DIMENSION BREAKDOWN")
print("="*90)
dim_analysis = sbti.analyze_dimensions(user_vector)
for group, data in dim_analysis.items():
    raw_str = " ".join([['L','M','H'][min(2, int(round(x/0.67)))] for x in data['raw']])
    print(f"{group:12} {data['level']:8} ({data['score']:5.1f}%) - {raw_str}")

print("\nTOP 5 MATCHES")
print("="*90)
for i, r in enumerate(results[:5], 1):
    marker = "[1]" if i == 1 else f"[{i}]"
    print(f"{marker} {r['code']:6} {r['score']:6.1f}% - {r['desc']}")

print("\nYOUR DNA:")
print("   " + " ".join(['L','M','H'][int(round(x/0.67))] for x in user_vector))
# Save result
result_data = {
    "timestamp": datetime.now().isoformat(),
    "main_type": final_type,
    "score": final_score,
    "confidence": confidence,
    "method": "Euclidean Distance (Improved)",
    "user_vector": user_vector.tolist(),
    "bonus_drink": bonus_drink,
    "top_matches": results[:5]
}

with open("sbt_result.json", "w", encoding="utf-8") as f:
    json.dump(result_data, f, ensure_ascii=False, indent=2)

print("\nResults saved to sbt_result.json")
print("Test completed!")

=== SBTI - Hybrid Method v2.4 (Improved Scoring) ===


QUESTION  1 / 31
Trong chuyện tình cảm, tôi thường cảm thấy mình không đủ tốt so với những người yêu cũ của người ấy.

   0 - Đúng vậy, tôi hay bị ám ảnh bởi điều đó.
   1 - Thỉnh thoảng tôi mới nghĩ vậy khi mọi thứ không suôn sẻ.
   2 - Không, tôi tin vào giá trị của mình ở hiện tại.
------------------------------------------------------------------------------------------
   OK - Saved for S1_SelfEsteem

QUESTION  2 / 31
Người yêu bạn rủ bạn về ra mắt gia đình, nhưng bạn biết gia đình họ có điều kiện và rất khó tính.

   0 - Sợ mình không đủ tốt, kiểu gì cũng bị chê.
   1 - Tôi lo một chút, chuẩn bị trước vài thứ để tự tin hơn.
   2 - Tôi tự tin mình sẽ ghi điểm.
------------------------------------------------------------------------------------------
   OK - Saved for S1_SelfEsteem

QUESTION  3 / 31
Tôi là ai? Tôi thích gì? Tôi giỏi gì? Tôi muốn gì? Đừng hỏi tôi...

   0 - Ủa, sao giống mình thế...
   1 - Mình đang đọc cái * th